In [ ]:
%load_ext autoreload

In [ ]:
import os
from pathlib import Path
import pickle
import random
import re
from typing import cast, Optional

import h5py
import pandas as pd
import seaborn as sns
import numpy as np
import mne
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
%autoreload 2

from src.data import add_metadata_features
from src.data_cleaning import prepare_ABC_results, compute_stimulus_correlation
from src.models import causal4
from src.viz import precompute_timit_bounds

In [ ]:
outdir = "outputs/causal4/compute_A_stimulus_correlations"

epochs_paths = list(Path("outputs/epochs_preprocessed").glob("*.fif"))
all_A_result_paths = list(Path("outputs/causal4/find_As").rglob("*_results.csv"))
all_A_decoders_path = list(Path("outputs/causal4/find_As").rglob("*_decoders.pt"))

A_unified_result_path = Path("outputs/causal4/unify_As/results.csv")
A_unified_decoders_path = Path("outputs/causal4/unify_As/unified_decoders.pt")

# all_B_result_paths = list(Path("outputs/causal4/find_Bs").glob("*_results.csv"))
B_annotated_path = Path("outputs/causal4/annotated_B_results.csv")

C_results_path = Path("outputs/causal4/find_Cs/C_study_results.csv")

In [ ]:
epochs = {re.findall(r"/([^/]+)_epo\.fif", str(epochs_path))[0]:
          mne.read_epochs(epochs_path, verbose=False)
            for epochs_path in tqdm(epochs_paths)}
for ep in epochs.values():
    ep.metadata = add_metadata_features(ep.metadata)

In [ ]:
A_results = pd.concat([pd.read_csv(p) for p in all_A_result_paths], ignore_index=True).query("A")

In [ ]:
A_decoders = {
    re.findall(r"/([^/]+)_decoders\.pt", str(p))[0]:
    torch.load(p) for p in tqdm(all_A_decoders_path)}

In [ ]:
A_results["stimulus_correlation"], A_outcomes = compute_stimulus_correlation(A_results, A_decoders, epochs, return_outcomes=True)

In [ ]:
A_results.to_csv(f"{outdir}/results.csv", index=False)